In [1]:
!pip install -q kaggle

In [3]:
!kaggle datasets download -d grassknoted/asl-alphabet -p /content/asl --unzip

Dataset URL: https://www.kaggle.com/datasets/grassknoted/asl-alphabet
License(s): GPL-2.0
100% 1.03G/1.03G [00:10<00:00, 101MB/s] 



In [4]:
from pathlib import Path

base = Path("/content/asl")

for p in base.rglob("asl_alphabet_train"):
    if p.is_dir():
        print("Dataset folder:", p)

Dataset folder: /content/asl/asl_alphabet_train
Dataset folder: /content/asl/asl_alphabet_train/asl_alphabet_train


In [5]:
import torch
import torch.nn as nn
from torchvision import models
from pathlib import Path
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split,Subset

In [6]:
raw = Path("/content/asl/asl_alphabet_train/asl_alphabet_train")

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset = datasets.ImageFolder(raw, transform=transform)

# 200 images from each class
indices = []

for class_index in range(len(dataset.classes)):
    class_indices = [
        i for i, (_, label) in enumerate(dataset.samples)
        if label == class_index
    ]
    indices.extend(class_indices[:200])

small_dataset = Subset(dataset, indices)

train_size = int(0.8 * len(small_dataset))
val_size = len(small_dataset) - train_size

train_dataset, val_dataset = random_split(
    small_dataset, [train_size, val_size]
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

num_classes = len(dataset.classes)

print("Classes:", dataset.classes)
print("Number of classes:", num_classes)
print("Images selected:", len(small_dataset))

Classes: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']
Number of classes: 29
Images selected: 5800


Load the pretrained VGG16 model

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [9]:
model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:03<00:00, 143MB/s]


In [10]:
# Freeze the pretrained feature-extraction layers
for param in model.features.parameters():
    param.requires_grad = False

# Replace the final layer for our 29 ASL classes
model.classifier[6] = nn.Linear(4096, num_classes)

model = model.to(device)

print("Device:", device)
print("Number of classes:", num_classes)

Device: cuda
Number of classes: 29


Loss function and optimizer

In [11]:
criterion=nn.CrossEntropyLoss()

optimizer=torch.optim.Adam(
    model.classifier.parameters(),
    lr=0.0001
)

Model Train

In [12]:
import time

epochs = 3

for epoch in range(epochs):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0
    start_time = time.time()

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    accuracy = 100 * correct / total
    elapsed = time.time() - start_time

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss: {epoch_loss:.4f} "
        f"Accuracy: {accuracy:.2f}% "
        f"Time: {elapsed:.2f}s"
    )

Epoch [1/3] Loss: 1.1416 Accuracy: 69.83% Time: 40.50s
Epoch [2/3] Loss: 0.0751 Accuracy: 98.08% Time: 38.45s
Epoch [3/3] Loss: 0.0254 Accuracy: 99.50% Time: 39.70s


Model Evaluation

In [13]:
model.eval()

correct = 0
total = 0
val_loss = 0.0

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        val_loss += loss.item() * images.size(0)
        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

print("Validation Loss:", round(val_loss / total, 4))
print("Validation Accuracy:", round(100 * correct / total, 2), "%")

Validation Loss: 0.0279
Validation Accuracy: 98.97 %


Save Trained Model

In [14]:
from pathlib import Path

Path("/content/results").mkdir(parents=True, exist_ok=True)

torch.save({
    "model_state_dict": model.state_dict(),
    "classes": dataset.classes
}, "/content/results/vgg16_asl.pth")

print("VGG16 model saved!")

VGG16 model saved!


In [15]:
from google.colab import files

files.download("/content/results/vgg16_asl.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>